1) Loading data and Preprocessing

In [ ]:
# Bandpass filter (8-30 Hz)
# Epoching (2.0 - 6.0 s )

import mne
import numpy as np
import matplotlib.pyplot as plt
from mne import Epochs, pick_types, events_from_annotations
from mne.io import read_raw_gdf
from sklearn.model_selection import cross_val_score, ShuffleSplit
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from mne.decoding import CSP

# Load data
filepath = 'A01T.gdf'
raw = mne.io.read_raw_gdf(filepath, preload=True)
print(raw.info)


Extracting GDF parameters from A09T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG, EOG-left, EOG-central, EOG-right
Creating raw.info structure...
Reading 0 ... 673327  =      0.000 ...  2693.308 secs...


d:\Program Files\Python\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


<Info | 8 non-empty values
 bads: []
 ch_names: EEG-Fz, EEG-0, EEG-1, EEG-2, EEG-3, EEG-4, EEG-5, EEG-C3, EEG-6, ...
 chs: 25 EEG
 custom_ref_applied: False
 highpass: 0.5 Hz
 lowpass: 100.0 Hz
 meas_date: 2004-11-16 12:00:00 UTC
 nchan: 25
 projs: []
 sfreq: 250.0 Hz
 subject_info: <subject_info | his_id: A09, sex: 0, last_name: X, birthday: 1987-11-16>
>


In [50]:
#Filter the raw signal with a band pass filter in 8-30 HZ
raw.filter(8., 30., method="fir")

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------


Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 413 samples (1.652 s)



<RawGDF | A09T.gdf, 25 x 673328 (2693.3 s), ~128.5 MiB, data loaded>

2. Epoching (2-6 seconds post-cue)+ Channel Selection (22 EEG channels)

In [51]:
fs = raw.info['sfreq']

# Get events
events,_ = mne.events_from_annotations(raw)

# Remove EOG channels and Pick EEG Chanels
raw.info['bads'] += ['EOG-right','EOG-central','EOG-left']
pick_raw = mne.pick_types (raw.info, meg=False, eeg=True, eog=False, stim=False, exclude='bads')
pick_raw

Used Annotations descriptions: [np.str_('1023'), np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('769'), np.str_('770'), np.str_('771'), np.str_('772')]


array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21])

In [52]:
# Create epochs for motor imagery trials
# Cue events are 769, 770, 771, 772
event_id= dict ({'769':7, '770':8 , '771':9, '772':10})
tmin, tmax = 2.0 , 6.0  

# Create epochs
epochs = mne.Epochs (raw, events, event_id, tmin, tmax, picks=pick_raw , preload=True, baseline= None)

print(f"\nNumber of epochs created: {len(epochs)}")
print(f"Epoch duration: {epochs.tmax - epochs.tmin} seconds")
print(f"Epoch shape: {epochs.get_data().shape} (epochs, channels, samples)")


Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 1001 original time points ...
1 bad epochs dropped

Number of epochs created: 287
Epoch duration: 4.0 seconds
Epoch shape: (287, 22, 1001) (epochs, channels, samples)


3) Common Spatial Patterns (CSP)

In [53]:
# Get data and labels
X = epochs.get_data()  # shape: (n_epochs, n_channels, n_times)
Y = epochs.events[:, -1]  # labels: 7,8,9,10
Y = Y - Y.min()

print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {Y.shape}")
print(f"Unique labels: {np.unique(Y)}")

Feature matrix shape: (287, 22, 1001)
Labels shape: (287,)
Unique labels: [0 1 2 3]


In [54]:
# CSP parameters
n_components = 16

csp = CSP( n_components=n_components,  reg=None,  log=True,  norm_trace=False)

X_csp = csp.fit_transform(X, Y)

print("CSP shape:", X_csp.shape)

Computing rank from data with rank=None
    Using tolerance 0.00015 (2.2e-16 eps * 22 dim * 3e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
CSP shape: (287, 16)


4. Feature extraction ( Bandpower + Shannon entropy)

In [55]:
#Alpha (8–13 Hz), Beta (13–30 Hz), Ratio
# 12 (CSP) + 3*22 (bandpower) + 1*22 (entropy) = 100 features per trial

from scipy.signal import welch
from scipy.stats import entropy
from scipy.signal import welch
from scipy.integrate import trapezoid

fs = 250

def bandpower(signal, fs, band):
    f, psd = welch(signal, fs=fs, nperseg=256)
    idx = (f >= band[0]) & (f <= band[1])
    return trapezoid(psd[idx], f[idx])

def shannon_entropy(signal):
    hist, _ = np.histogram(signal, bins=50, density=True)
    hist = hist[hist > 0]
    return entropy(hist)

In [56]:
X_features = []

for i in range(len(X)):

    trial_features = []

    # ---- CSP FEATURES ----
    trial_features.extend(X_csp[i])  # 12 features

    # ---- RAW EEG FEATURES (22 Channels) ----
    for ch in range(22):

        sig = X[i, ch, :]

        # Bandpower
        alpha = bandpower(sig, fs, (8, 13))

        # Entropy
        ent = shannon_entropy(sig)

        trial_features.extend([alpha,ent])

    X_features.append(trial_features)

X_features = np.array(X_features)

print("Final feature shape:", X_features.shape)

Final feature shape: (287, 60)


5) Classification (Compare LDA, SVM, XGBoost and MLP)

In [57]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

LDA = LinearDiscriminantAnalysis()

# SVM classifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# SVM needs scaling!
scaler = StandardScaler()
X_csp_scaled = scaler.fit_transform(X_csp)
SVM = SVC(kernel='rbf', C=1.0, gamma='scale')

RandomForest= RandomForestClassifier( n_estimators=300, random_state=42)

XGBoost= XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softmax",
        num_class=4,
        eval_metric="mlogloss"
    )

MLP= MLPClassifier(  hidden_layer_sizes=(64, 32),  max_iter=1000,  random_state=42)


6) Cross-Validation (ShuffleSplit with 10 Splits)

In [58]:
from sklearn.model_selection import ShuffleSplit
from sklearn.metrics import accuracy_score
import numpy as np
from sklearn.preprocessing import StandardScaler

models = {
    "LDA": LDA,
    "SVM": SVM,
    "RandomForest": RandomForest,
    "XGBoost": XGBoost,
    "MLP": MLP
}

cv = ShuffleSplit(
    n_splits=10,
    test_size=0.2,
    random_state=42
)

results = {}

for model_name, model in models.items():

    accuracies = []

    print(f"\nRunning {model_name}...")

    for train_idx, test_idx in cv.split(X_features, Y):

        X_train, X_test = X_features[train_idx], X_features[test_idx]
        Y_train, Y_test = Y[train_idx], Y[test_idx]
        
        
        model.fit(X_train, Y_train)
        Y_pred = model.predict(X_test)

        acc = accuracy_score(Y_test, Y_pred)

        accuracies.append(acc)

    results[model_name] = {
        "Mean Accuracy": np.mean(accuracies),
        "Std Accuracy": np.std(accuracies),
        "Best Fold": np.max(accuracies)
    }


Running LDA...

Running SVM...

Running RandomForest...

Running XGBoost...

Running MLP...


In [59]:
print("\n" + "="*60)
print(f"{'Classifier':<15} {'Mean Acc':<12} {'Std':<10} {'Best Fold':<10}")
print("="*60)

for model_name, metrics in results.items():

    print(
        f"{model_name:<15}"
        f"{metrics['Mean Accuracy']:.3f}      "
        f"{metrics['Std Accuracy']:.3f}      "
        f"{metrics['Best Fold']:.3f}"
    )

print("="*60)


Classifier      Mean Acc     Std        Best Fold 
LDA            0.364      0.069      0.500
SVM            0.221      0.055      0.345
RandomForest   0.328      0.034      0.379
XGBoost        0.353      0.054      0.448
MLP            0.428      0.039      0.483
